# 05: Goal prediction models

Predict the expected goals for the home and away team in each match. Models are compared on the 2024/25 validation season against a simple baseline: the training average of home and away goals. The 2025/26 test season is not used in this notebook.

**Input:** `data/processed/model_table.csv`, built in `04_training_table.ipynb` (one row per match, with the home team's and the away team's pre-match form side by side).

**Approach:** predict goals as *expected values*, not exact scores. Football scores are low and noisy, so the useful output is "this side is expected to score 1.7 goals", which a later step turns into scoreline and win/draw/loss probabilities.

**Contents**
1. Load the data and define the split
2. Baseline: the same goals for every match
3. Poisson regression
4. Results on the validation season
5. Sanity checks
6. What the model learned (coefficients)
7. Summary and next steps

## 1. Load the data and define the split

**Loading.** A CSV file does not remember data types, so `Date` is parsed back into a date and `Season` is kept as text. Season codes such as `"2324"` are compared as text in the split below, so reading them as numbers would break the logic.

**Rest days.** Days of rest is normally 3 to 7. After the summer break it jumps to 80 or more, which is a different situation and would distort the model, so values are capped at 14 (anything above is treated as "well rested").

**Features.** For each side the model sees points over the last 5 matches, average goals scored and conceded over the last 5 and over the last 10 matches, and days of rest. All of these come from the `team_form` view and use only matches played *before* the one being predicted. The targets (`home_goals`, `away_goals`), the bookmaker odds and any statistic from the match itself (shots, fouls, cards) are deliberately left out of the feature list. That list is the project's defence against data leakage.

**Time-based split, no shuffling.**
- Train: seasons up to 2023/24, where the model learns its numbers
- Validation: 2024/25, for comparing models and choosing settings
- Test: 2025/26, touched once at the very end so its score is an honest estimate

Random shuffling would let the model learn from matches played after the ones it predicts, which never happens in real use.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance

df = pd.read_csv("../data/processed/model_table.csv",
                 parse_dates=["Date"], dtype={"Season": str})

# Days of rest after the summer break are huge and meaningless, so cap them
df["h_rest"] = df["h_rest"].clip(upper=14)
df["a_rest"] = df["a_rest"].clip(upper=14)

features = [
    "h_points_last5", "h_gf5", "h_ga5", "h_gf10", "h_ga10", "h_rest",
    "a_points_last5", "a_gf5", "a_ga5", "a_gf10", "a_ga10", "a_rest",
]

train = df[df["Season"] <= "2324"]
valid = df[df["Season"] == "2425"]
test  = df[df["Season"] == "2526"]    # not used until the final evaluation

print(len(train), len(valid), len(test))
df[features].isna().sum()             # should be all 0 after the earlier filter

2560 375 375


h_points_last5    0
h_gf5             0
h_ga5             0
h_gf10            0
h_ga10            0
h_rest            0
a_points_last5    0
a_gf5             0
a_ga5             0
a_gf10            0
a_ga10            0
a_rest            0
dtype: int64

**Output:** 2,560 training, 375 validation and 375 test matches, and no missing values in any feature. A full season has 380 matches, so 5 are missing from each of the validation and test seasons. They were removed earlier because a team had fewer than 5 prior matches, most likely the first games of a newly promoted club with no history in the data.

## 2. Baseline: the same goals for every match

The simplest possible model ignores the teams completely and predicts the training-set average of home goals and away goals for every match. It is computed on the training seasons only, because using validation data here would leak information.

A baseline matters because a model's score means little on its own. "Off by 0.99 goals on average" only becomes meaningful next to "the do-nothing approach is off by 1.05".

In [2]:
base_home = train["home_goals"].mean()
base_away = train["away_goals"].mean()
print(round(base_home, 2), round(base_away, 2))

1.56 1.29


**Result:** the training average is 1.56 home goals and 1.29 away goals per match. The gap of about 0.27 goals is home advantage expressed in goals.

### Evaluation helper and baseline scores

One function scores every model the same way, so comparisons are fair. Two metrics are computed separately for home and away goals:

- **MAE (mean absolute error):** the average distance between predicted and real goals. Easy to read, and lower is better.
- **Poisson deviance:** a score designed for count data such as goals. It penalises an error relative to the size of the prediction (missing by 1 goal when 0.3 was expected is worse than missing by 1 when 3 was expected). Lower is better, and this is the more reliable of the two metrics here.

`np.full(len(valid), base_home)` repeats the baseline number once per validation match, because the helper expects one prediction per match.

In [3]:
def evaluate(name, pred_home, pred_away, data):
    return {
        "model": name,
        "MAE home": mean_absolute_error(data["home_goals"], pred_home),
        "MAE away": mean_absolute_error(data["away_goals"], pred_away),
        "Poisson dev home": mean_poisson_deviance(data["home_goals"], pred_home),
        "Poisson dev away": mean_poisson_deviance(data["away_goals"], pred_away),
    }

results = [evaluate("Baseline (training average)",
                    np.full(len(valid), base_home),
                    np.full(len(valid), base_away),
                    valid)]
pd.DataFrame(results).round(3)

,model,MAE home,MAE away,Poisson dev home,Poisson dev away
0,Baseline (training average),1.05,0.929,1.239,1.142


**Baseline on the validation season:** MAE 1.050 (home) and 0.929 (away), Poisson deviance 1.239 (home) and 1.142 (away). These are the numbers any model has to beat.

## 3. Poisson regression

**Why Poisson?** Goals are small whole numbers (0, 1, 2, 3...) that cannot be negative, and they behave roughly like a Poisson distribution, which describes how often a rare event happens in a fixed window. An ordinary linear regression can predict negative or nonsensical values, while Poisson regression predicts an expected goal count that is always positive. Internally it models expected goals as `exp(weighted sum of features)`, so each feature multiplies the expectation up or down.

**The pipeline** chains two steps that always run in the same order:
- `StandardScaler` rescales each feature to an average of 0 and a spread of 1, so that features on different scales (points up to 15, goals around 1) are treated evenly by the regularisation. It learns its scaling from the training data only, and the same numbers are reused on validation.
- `PoissonRegressor(alpha=1.0)` is the model. `alpha` is the regularisation strength, a penalty that keeps coefficients small. It helps here because the last-5 and last-10 features overlap heavily.

**Two separate models**, one for home goals and one for away goals, because what drives each side's scoring is not identical.

The predictions are *expected goals*, not scorelines. A prediction of 1.7 does not mean the model expects exactly 1.7 goals, it means that is the long-run average for matches that look like this one.

In [4]:
def fit_poisson(target):
    model = make_pipeline(StandardScaler(), PoissonRegressor(alpha=1.0, max_iter=1000))
    model.fit(train[features], train[target])
    return model

model_home = fit_poisson("home_goals")
model_away = fit_poisson("away_goals")

pred_home = model_home.predict(valid[features])
pred_away = model_away.predict(valid[features])

results.append(evaluate("Poisson regression", pred_home, pred_away, valid))
pd.DataFrame(results).round(3)

,model,MAE home,MAE away,Poisson dev home,Poisson dev away
0,Baseline (training average),1.050,0.929,1.239,1.142
1,Poisson regression,0.993,0.888,1.172,1.071


## 4. Results on the validation season (2024/25)

| Metric | Baseline | Poisson regression | Improvement |
|---|---|---|---|
| MAE, home goals | 1.050 | 0.993 | 5.4% |
| MAE, away goals | 0.929 | 0.888 | 4.4% |
| Poisson deviance, home | 1.239 | 1.172 | 5.4% |
| Poisson deviance, away | 1.142 | 1.071 | 6.2% |

Poisson regression beats the baseline on all four measures, by roughly 4 to 6 percent. That is a modest gain, which is normal: football results contain a lot of luck, and recent form explains only part of it. A consistent improvement across both sides and both metrics is still evidence that the features carry real information about goals.

**Caveat:** this is a single validation season of 375 matches, so a difference of a few percent could partly be noise. The final judgement will come from the test season and from comparison with other models.

## 5. Sanity checks

If the model's average prediction is far from the real average, something is wrong, for example a scaling or leakage problem. Close agreement means the model is calibrated at the level of the whole season.

In [5]:
print(pred_home.mean(), valid["home_goals"].mean())   # should be close
print(pred_away.mean(), valid["away_goals"].mean())   # should be close

1.5857660571461862 1.5173333333333334
1.2992861680016383 1.4266666666666667


| | Average predicted | Average actual |
|---|---|---|
| Home goals | 1.586 | 1.517 |
| Away goals | 1.299 | 1.427 |

Home goals are slightly over-predicted (by about 0.07) and away goals under-predicted (by about 0.13). A likely reason is visible in the baseline step: 2024/25 had more away goals per match (1.43) than the training average (1.29), so any model fitted on earlier seasons will sit a little low. The baseline carries the same shift, so it does not explain the model's improvement. With 375 matches the gap is roughly two standard errors, so it may be partly chance and partly a real change in scoring. It is worth keeping in mind for the later evaluation.

## 6. What the model learned (coefficients)

`model_home[-1]` picks the last pipeline step, the regression itself. Because the features were scaled, each coefficient is the change in log expected home goals for a one-standard-deviation increase in that feature. A coefficient of 0.10 corresponds to roughly a 10% increase in expected goals, since `exp(0.10)` is about 1.105. Only the home-goals model is inspected here.

In [6]:
coefs = pd.Series(model_home[-1].coef_, index=features).sort_values()
coefs

a_gf10           -0.040973
h_ga10           -0.036508
a_gf5            -0.028779
a_points_last5   -0.012339
h_ga5            -0.011620
h_rest           -0.006988
a_rest            0.011549
a_ga5             0.019323
h_points_last5    0.037667
h_gf5             0.043626
a_ga10            0.053936
h_gf10            0.090017
dtype: float64

**Reading the coefficients (home-goals model):**

- **Signs make football sense.** The home team's own scoring (`h_gf10` +0.090, `h_gf5` +0.044) and recent points (`h_points_last5` +0.038) raise its expected goals. The away team's goals conceded (`a_ga10` +0.054, `a_ga5` +0.019) also raise them, since a leaky defence lets more in.
- **Strong opponents lower home scoring.** A better-scoring away side (`a_gf10` -0.041, `a_gf5` -0.029) or one in good form (`a_points_last5` -0.012) goes with fewer home goals. Likewise, a home team that concedes a lot (`h_ga10` -0.037) tends to be weaker overall and scores less. A plausible reading is that these features act as proxies for team quality, but the model only sees correlations.
- **Effects are modest.** The largest, `h_gf10`, means about +9% expected home goals for a one-standard-deviation increase (`exp(0.090)` is about 1.09). Ten of the twelve coefficients are smaller than 0.05 in size.
- **Rest days barely matter** (`h_rest` -0.007, `a_rest` +0.012).
- **Do not over-read single coefficients.** The last-5 and last-10 features overlap, so they share credit, and regularisation shrinks everything toward zero.

## 7. Summary and next steps

**What was done**
- Loaded the one-row-per-match table and split it by time: train up to 2023/24, validation 2024/25, test 2025/26 (untouched)
- Built a baseline (training-average goals) and two Poisson regression models, one per side, using only pre-match form features
- Compared them on the validation season with MAE and Poisson deviance

**Main finding:** Poisson regression improves on the baseline by roughly 4 to 6 percent on every metric, and its coefficients have sensible signs. The gain is modest, as expected for football.

**Limitations**
- Only recent-form features: no team-strength rating, and no player, injury or lineup information
- One validation season of 375 matches, so results are noisy
- The validation season had more away goals than the training average, so predictions sit slightly low for away teams
- A linear model on the log scale, with no interaction between the two teams' features
- The first matches of newly promoted clubs are excluded for lack of history

**Next steps**
1. Convert each match's two expected-goals values into a scoreline probability grid, and sum it into home win, draw and away win probabilities
2. Evaluate result accuracy and log loss against "always home win" and the bookmaker odds
3. Optionally add LightGBM with a Poisson objective as a second model
4. Run the final evaluation once on the 2025/26 test season
5. Save the trained models and each team's latest form for the Streamlit demo